# Experiment 04 — Video Swin Regularization

This notebook documents the regularization experiment on the 10-class cricket-shot dataset.

- Backbone: pretrained Swin3D-T (Kinetics-400)
- Input: 32 frames, 224 × 224
- Regularization: cross-entropy label smoothing (`0.1`)
- Best checkpoint: epoch 5 (validation accuracy 90.11%)
- Final test: 88.38% (251/284 correct)

The original notebook included earlier baseline training and unrelated debugging. This version retains the regularization experiment, its training logs, restart/recovery steps, and final evaluation. Kaggle paths and checkpoint restore steps may require adjustment in a fresh environment.


In [ ]:
import os
import time
import json
import random

import cv2
import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models.video import (
    swin3d_t,
    Swin3D_T_Weights
)


print("=" * 60)
print("VIDEO SWIN — REGULARIZATION EXPERIMENT")
print("=" * 60)

print("PyTorch version    :", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("GPU count          :", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(
            f"GPU {i}:",
            torch.cuda.get_device_name(i)
        )


# Use GPU 0 only for fair comparison
device = torch.device(
    "cuda:0" if torch.cuda.is_available()
    else "cpu"
)

print("\nUsing device:", device)

if torch.cuda.is_available():
    print(
        "Using GPU   :",
        torch.cuda.get_device_name(0)
    )

VIDEO SWIN — REGULARIZATION EXPERIMENT
PyTorch version    : 2.11.0+cu128
Torchvision version: 0.26.0+cu128
CUDA available     : True
GPU count          : 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Using device: cuda:0
Using GPU   : Tesla T4


In [ ]:
# ============================================================
# CELL 2 — DATASET PATH + CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Dataset path
# ------------------------------------------------------------

DATASET_ROOT = (
    "/kaggle/input/datasets/"
    "nivrithareddykunduru/"
    "cricketshot-dataset/dataset"
)

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")


print("=" * 60)
print("DATASET PATH CHECK")
print("=" * 60)

print("Dataset root:", DATASET_ROOT)
print("Train exists:", os.path.exists(TRAIN_DIR))
print("Val exists  :", os.path.exists(VAL_DIR))
print("Test exists :", os.path.exists(TEST_DIR))


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 27

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Video configuration
# ------------------------------------------------------------

NUM_FRAMES = 32
IMAGE_SIZE = 224
NUM_CLASSES = 10


# ------------------------------------------------------------
# Classes
# ------------------------------------------------------------

CLASSES = [
    "cover",
    "defense",
    "flick",
    "hook",
    "late_cut",
    "lofted",
    "pull",
    "square_cut",
    "straight",
    "sweep"
]

CLASS_TO_IDX = {
    class_name: idx
    for idx, class_name in enumerate(CLASSES)
}


print("\n" + "=" * 60)
print("EXPERIMENT CONFIG")
print("=" * 60)

print("Seed        :", SEED)
print("Num frames  :", NUM_FRAMES)
print("Image size  :", IMAGE_SIZE)
print("Num classes :", NUM_CLASSES)

print("\nClass mapping:")

for class_name, idx in CLASS_TO_IDX.items():
    print(f"{idx}: {class_name}")

DATASET PATH CHECK
Dataset root: /kaggle/input/datasets/nivrithareddykunduru/cricketshot-dataset/dataset
Train exists: False
Val exists  : False
Test exists : False

EXPERIMENT CONFIG
Seed        : 27
Num frames  : 32
Image size  : 224
Num classes : 10

Class mapping:
0: cover
1: defense
2: flick
3: hook
4: late_cut
5: lofted
6: pull
7: square_cut
8: straight
9: sweep


In [ ]:
# ============================================================
# CELL 3 — PREPROCESSING + UNIFORM TEMPORAL SAMPLING
# ============================================================

from torchvision.models.video import Swin3D_T_Weights


# ------------------------------------------------------------
# 1. Pretrained Kinetics-400 weights
# ------------------------------------------------------------

weights = Swin3D_T_Weights.KINETICS400_V1

preprocess = weights.transforms()


# Same preprocessing for ALL splits
train_transform = preprocess
val_transform   = preprocess
test_transform  = preprocess


print("=" * 60)
print("PREPROCESSING")
print("=" * 60)

print(weights)
print("\nTransform:")
print(preprocess)


# ------------------------------------------------------------
# 2. Uniform temporal sampling
# ------------------------------------------------------------

def uniform_temporal_indices(
    total_frames,
    num_frames=32
):

    if total_frames <= 0:
        return np.zeros(
            num_frames,
            dtype=np.int64
        )

    indices = np.linspace(
        0,
        total_frames - 1,
        num_frames
    )

    indices = np.round(
        indices
    ).astype(np.int64)

    return indices


# ------------------------------------------------------------
# 3. Quick sanity check
# ------------------------------------------------------------

example_total_frames = 64

indices_1 = uniform_temporal_indices(
    example_total_frames,
    NUM_FRAMES
)

indices_2 = uniform_temporal_indices(
    example_total_frames,
    NUM_FRAMES
)


print("\n" + "=" * 60)
print("TEMPORAL SAMPLING CHECK")
print("=" * 60)

print("Total frames :", example_total_frames)
print("Selected     :", len(indices_1))

print("\nIndices:")
print(indices_1)

print(
    "\nDeterministic:",
    np.array_equal(indices_1, indices_2)
)

print(
    "Sorted       :",
    np.all(np.diff(indices_1) >= 0)
)

PREPROCESSING
Swin3D_T_Weights.KINETICS400_V1

Transform:
VideoClassification(
    crop_size=[224, 224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

TEMPORAL SAMPLING CHECK
Total frames : 64
Selected     : 32

Indices:
[ 0  2  4  6  8 10 12 14 16 18 20 22 24 26 28 30 33 35 37 39 41 43 45 47
 49 51 53 55 57 59 61 63]

Deterministic: True
Sorted       : True


In [ ]:
# ============================================================
# CELL 4 — CRICKET VIDEO DATASET
# ============================================================

class CricketVideoDataset(Dataset):

    def __init__(
        self,
        root_dir,
        transform,
        num_frames=32
    ):

        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames

        self.samples = []

        valid_extensions = (
            ".avi",
            ".mp4",
            ".mov",
            ".mkv"
        )


        # ----------------------------------------------------
        # Collect videos
        # ----------------------------------------------------

        for class_name in CLASSES:

            class_dir = os.path.join(
                root_dir,
                class_name
            )

            if not os.path.isdir(class_dir):
                continue

            label = CLASS_TO_IDX[class_name]

            for filename in sorted(
                os.listdir(class_dir)
            ):

                if filename.lower().endswith(
                    valid_extensions
                ):

                    video_path = os.path.join(
                        class_dir,
                        filename
                    )

                    self.samples.append(
                        (
                            video_path,
                            label
                        )
                    )


        print(
            f"Loaded {len(self.samples)} videos "
            f"from {root_dir}"
        )


    def __len__(self):

        return len(self.samples)


    def __getitem__(self, index):

        video_path, label = self.samples[index]

        cap = cv2.VideoCapture(video_path)

        total_frames = int(
            cap.get(cv2.CAP_PROP_FRAME_COUNT)
        )


        # ----------------------------------------------------
        # Uniform sampling — SAME for every split
        # ----------------------------------------------------

        frame_indices = uniform_temporal_indices(
            total_frames,
            self.num_frames
        )


        frames = []


        # ----------------------------------------------------
        # Read selected frames
        # ----------------------------------------------------

        for frame_idx in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_idx)
            )

            success, frame = cap.read()

            if success:

                frame = cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB
                )

                frames.append(frame)


        cap.release()


        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if len(frames) == 0:

            raise RuntimeError(
                f"Could not decode video: {video_path}"
            )


        # If some selected frames failed to decode,
        # repeat the last valid frame.
        while len(frames) < self.num_frames:

            frames.append(
                frames[-1].copy()
            )


        # ----------------------------------------------------
        # Convert to tensor
        #
        # [T, H, W, C]
        #       ↓
        # [T, C, H, W]
        # ----------------------------------------------------

        frames = np.stack(
            frames,
            axis=0
        )

        frames = torch.from_numpy(
            frames
        ).permute(
            0, 3, 1, 2
        )


        # ----------------------------------------------------
        # Official Kinetics preprocessing
        #
        # Input : [T,C,H,W]
        # Output: [C,T,H,W]
        # ----------------------------------------------------

        if self.transform is not None:

            frames = self.transform(
                frames
            )


        return frames, label

In [ ]:
# ============================================================
# CELL 5 — DATASETS + DATALOADERS
# ============================================================

train_dataset = CricketVideoDataset(
    TRAIN_DIR,
    transform=train_transform,
    num_frames=NUM_FRAMES
)

val_dataset = CricketVideoDataset(
    VAL_DIR,
    transform=val_transform,
    num_frames=NUM_FRAMES
)

test_dataset = CricketVideoDataset(
    TEST_DIR,
    transform=test_transform,
    num_frames=NUM_FRAMES
)


BATCH_SIZE = 2


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("\n" + "=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print("Train videos :", len(train_dataset))
print("Val videos   :", len(val_dataset))
print("Test videos  :", len(test_dataset))

print()

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

Loaded 1321 videos from /kaggle/input/datasets/nivrithareddykunduru/cricketshot-dataset/dataset/train
Loaded 283 videos from /kaggle/input/datasets/nivrithareddykunduru/cricketshot-dataset/dataset/val
Loaded 284 videos from /kaggle/input/datasets/nivrithareddykunduru/cricketshot-dataset/dataset/test

DATASET SUMMARY
Train videos : 1321
Val videos   : 283
Test videos  : 284

Train batches: 661
Val batches  : 142
Test batches : 142


In [ ]:
# ============================================================
# CELL 6 — DATASET SANITY CHECK
# ============================================================

print("=" * 60)
print("DATASET SANITY CHECK")
print("=" * 60)


# Load same training sample twice
video_1, label_1 = train_dataset[0]
video_2, label_2 = train_dataset[0]


print("Video shape :", video_1.shape)
print("Data type   :", video_1.dtype)
print("Label       :", label_1)
print("Class       :", CLASSES[label_1])


# Difference between two loads
mean_difference = torch.mean(
    torch.abs(video_1 - video_2)
).item()


print("\nMean difference   :", mean_difference)

print(
    "Outputs identical:",
    torch.equal(video_1, video_2)
)


# Additional checks
print("\nNaN present :", torch.isnan(video_1).any().item())
print("Inf present :", torch.isinf(video_1).any().item())


# ------------------------------------------------------------
# Test one complete batch
# ------------------------------------------------------------

videos, labels = next(iter(train_loader))

print("\n" + "=" * 60)
print("BATCH CHECK")
print("=" * 60)

print("Batch video shape :", videos.shape)
print("Batch label shape :", labels.shape)
print("Batch dtype       :", videos.dtype)
print("Labels            :", labels)

DATASET SANITY CHECK
Video shape : torch.Size([3, 32, 224, 224])
Data type   : torch.float32
Label       : 0
Class       : cover

Mean difference   : 0.0
Outputs identical: True

NaN present : False
Inf present : False

BATCH CHECK
Batch video shape : torch.Size([2, 3, 32, 224, 224])
Batch label shape : torch.Size([2])
Batch dtype       : torch.float32
Labels            : tensor([4, 8])


In [ ]:
# ============================================================
# CELL 7 — FRESH PRETRAINED SWIN3D-T
# ============================================================

import torch.nn as nn
from torchvision.models.video import (
    swin3d_t,
    Swin3D_T_Weights
)


print("=" * 60)
print("LOADING FRESH SWIN3D-T")
print("=" * 60)


# ------------------------------------------------------------
# 1. Kinetics-400 pretrained weights
# ------------------------------------------------------------

weights = Swin3D_T_Weights.KINETICS400_V1


# ------------------------------------------------------------
# 2. Load fresh pretrained model
# ------------------------------------------------------------

model = swin3d_t(
    weights=weights
)


# ------------------------------------------------------------
# 3. Replace original Kinetics classifier
#    400 classes → our 10 cricket classes
# ------------------------------------------------------------

in_features = model.head.in_features

print("Original head input features:", in_features)

model.head = nn.Linear(
    in_features,
    NUM_CLASSES
)


# ------------------------------------------------------------
# 4. Move model to GPU
# ------------------------------------------------------------

model = model.to(device)


# ------------------------------------------------------------
# 5. Parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("\nModel            : Swin3D-T")
print("Pretrained       : KINETICS400_V1")
print("Classes          :", NUM_CLASSES)

print(
    f"Total parameters : "
    f"{total_params:,}"
)

print(
    f"Trainable params : "
    f"{trainable_params:,}"
)


# ------------------------------------------------------------
# 6. Test forward pass
# ------------------------------------------------------------

model.eval()

sample_batch, sample_labels = next(
    iter(train_loader)
)

sample_batch = sample_batch.to(device)


with torch.no_grad():

    sample_output = model(
        sample_batch
    )


print("\nInput shape :", sample_batch.shape)
print("Output shape:", sample_output.shape)


# Return to training mode
model.train()

LOADING FRESH SWIN3D-T
Downloading: "https://download.pytorch.org/models/swin3d_t-7615ae03.pth" to /root/.cache/torch/hub/checkpoints/swin3d_t-7615ae03.pth


100%|██████████| 122M/122M [00:01<00:00, 105MB/s]  


Original head input features: 768

Model            : Swin3D-T
Pretrained       : KINETICS400_V1
Classes          : 10
Total parameters : 27,858,160
Trainable params : 27,858,160

Input shape : torch.Size([2, 3, 32, 224, 224])
Output shape: torch.Size([2, 10])


SwinTransformer3d(
  (patch_embed): PatchEmbed3d(
    (proj): Conv3d(3, 96, kernel_size=(2, 4, 4), stride=(2, 4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (features): Sequential(
    (0): Sequential(
      (0): SwinTransformerBlock(
        (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (attn): ShiftedWindowAttention3d(
          (qkv): Linear(in_features=96, out_features=288, bias=True)
          (proj): Linear(in_features=96, out_features=96, bias=True)
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (0): Linear(in_features=96, out_features=384, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=384, out_features=96, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
     

In [ ]:
# ============================================================
# CELL 8 — OPTIMIZER + LABEL SMOOTHING + AMP
# ============================================================

# ------------------------------------------------------------
# Hyperparameters
# ------------------------------------------------------------

BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.1


# ------------------------------------------------------------
# Separate backbone and classification head parameters
# ------------------------------------------------------------

backbone_params = []

for name, param in model.named_parameters():

    if not name.startswith("head."):
        backbone_params.append(param)


head_params = list(
    model.head.parameters()
)


# ------------------------------------------------------------
# AdamW optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(

    [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR
        },

        {
            "params": head_params,
            "lr": HEAD_LR
        }
    ],

    weight_decay=WEIGHT_DECAY
)


# ------------------------------------------------------------
# ⭐ REGULARIZATION CHANGE
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)


# ------------------------------------------------------------
# Automatic Mixed Precision
# ------------------------------------------------------------

scaler = torch.amp.GradScaler(
    "cuda"
)


# ------------------------------------------------------------
# Verify setup
# ------------------------------------------------------------

print("=" * 60)
print("REGULARIZATION TRAINING SETUP")
print("=" * 60)

print("Optimizer       : AdamW")
print("Backbone LR     :", optimizer.param_groups[0]["lr"])
print("Head LR         :", optimizer.param_groups[1]["lr"])
print("Weight decay    :", WEIGHT_DECAY)

print()

print("Loss            : CrossEntropyLoss")
print("Label smoothing :", LABEL_SMOOTHING)

print()

print("AMP enabled     :", scaler.is_enabled())

print(
    "Backbone params :",
    f"{sum(p.numel() for p in backbone_params):,}"
)

print(
    "Head params     :",
    f"{sum(p.numel() for p in head_params):,}"
)

print(
    "Total optimized :",
    f"{sum(p.numel() for p in backbone_params) + sum(p.numel() for p in head_params):,}"
)

REGULARIZATION TRAINING SETUP
Optimizer       : AdamW
Backbone LR     : 1e-05
Head LR         : 0.0001
Weight decay    : 0.0001

Loss            : CrossEntropyLoss
Label smoothing : 0.1

AMP enabled     : True
Backbone params : 27,850,470
Head params     : 7,690
Total optimized : 27,858,160


In [ ]:
# ============================================================
# CELL 9 — CHECKPOINT + HISTORY SETUP
# ============================================================

SAVE_DIR = "/kaggle/working/video_swin_regularization"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


LATEST_PATH = os.path.join(
    SAVE_DIR,
    "regularization_latest.pth"
)

BEST_PATH = os.path.join(
    SAVE_DIR,
    "regularization_best.pth"
)

HISTORY_PATH = os.path.join(
    SAVE_DIR,
    "regularization_history.json"
)

CONFIG_PATH = os.path.join(
    SAVE_DIR,
    "regularization_config.json"
)


# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0


# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

config = {

    "experiment": "label_smoothing_regularization",

    "model": "swin3d_t",
    "pretrained": "KINETICS400_V1",

    "num_classes": NUM_CLASSES,
    "num_frames": NUM_FRAMES,
    "image_size": IMAGE_SIZE,

    "batch_size": BATCH_SIZE,

    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,

    "weight_decay": WEIGHT_DECAY,

    "label_smoothing": LABEL_SMOOTHING,

    "temporal_sampling": "uniform_linspace",

    "spatial_preprocessing":
        "KINETICS400_V1_default",

    "seed": SEED,

    "classes": CLASSES
}


# Save configuration immediately
with open(CONFIG_PATH, "w") as f:
    json.dump(
        config,
        f,
        indent=4
    )


print("=" * 60)
print("CHECKPOINT SETUP")
print("=" * 60)

print("Save directory :", SAVE_DIR)
print("Latest         :", LATEST_PATH)
print("Best           :", BEST_PATH)
print("History        :", HISTORY_PATH)
print("Config         :", CONFIG_PATH)

print("\nBest val accuracy:", best_val_acc)

CHECKPOINT SETUP
Save directory : /kaggle/working/video_swin_regularization
Latest         : /kaggle/working/video_swin_regularization/regularization_latest.pth
Best           : /kaggle/working/video_swin_regularization/regularization_best.pth
History        : /kaggle/working/video_swin_regularization/regularization_history.json
Config         : /kaggle/working/video_swin_regularization/regularization_config.json

Best val accuracy: 0.0


In [ ]:
# ============================================================
# CELL 10 — TRAINING FUNCTIONS
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0


    for batch_idx, (videos, labels) in enumerate(
        loader,
        start=1
    ):

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)

            loss = criterion(
                outputs,
                labels
            )


        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()


        # Statistics
        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


        if batch_idx % 50 == 0:

            print(
                f"Batch {batch_idx}/{len(loader)} "
                f"| Loss: {loss.item():.4f}"
            )


    epoch_loss = (
        running_loss / total
    )

    epoch_acc = (
        correct / total
    )


    return epoch_loss, epoch_acc



# ============================================================
# VALIDATION
# ============================================================

@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0


    for videos, labels in loader:

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)

            loss = criterion(
                outputs,
                labels
            )


        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


    epoch_loss = (
        running_loss / total
    )

    epoch_acc = (
        correct / total
    )


    return epoch_loss, epoch_acc

In [ ]:
# ============================================================
# CELL 11 — SAVE TRAINING STATE
# ============================================================

def save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=False
):

    checkpoint = {

        "epoch": epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "best_val_acc":
            best_val_acc,

        "history":
            history,

        "config":
            config,

        "classes":
            CLASSES
    }


    # Always save latest
    torch.save(
        checkpoint,
        LATEST_PATH
    )


    # Save best separately
    if is_best:

        torch.save(
            checkpoint,
            BEST_PATH
        )


    # Save readable history
    with open(
        HISTORY_PATH,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=4
        )


    # Save config
    with open(
        CONFIG_PATH,
        "w"
    ) as f:

        json.dump(
            config,
            f,
            indent=4
        )

In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 1
# Label Smoothing = 0.1
# ============================================================

epoch = 1

print("\n" + "=" * 60)
print(f"REGULARIZATION — EPOCH {epoch}")
print("=" * 60)

print(f"Label smoothing: {LABEL_SMOOTHING}")

print(
    f"Starting LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"Starting LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

start_time = time.time()


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


# ------------------------------------------------------------
# VALIDATE
# ------------------------------------------------------------

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


# ------------------------------------------------------------
# UPDATE HISTORY
# ------------------------------------------------------------

history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


# ------------------------------------------------------------
# CHECK BEST
# ------------------------------------------------------------

is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

elapsed_minutes = (
    time.time() - start_time
) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(f"Label Smooth: {LABEL_SMOOTHING}")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


REGULARIZATION — EPOCH 1
Label smoothing: 0.1
Starting LR backbone: 1.00e-05
Starting LR head    : 1.00e-04
Batch 50/661 | Loss: 2.0819
Batch 100/661 | Loss: 2.2591
Batch 150/661 | Loss: 2.3821
Batch 200/661 | Loss: 1.8021
Batch 250/661 | Loss: 1.8986
Batch 300/661 | Loss: 1.9223
Batch 350/661 | Loss: 2.3545
Batch 400/661 | Loss: 1.1238
Batch 450/661 | Loss: 1.0363
Batch 500/661 | Loss: 1.9135
Batch 550/661 | Loss: 1.3742
Batch 600/661 | Loss: 1.4273
Batch 650/661 | Loss: 1.5115

⭐ New best checkpoint saved

Train Loss : 1.8182
Train Acc  : 41.26%
Val Loss   : 1.3273
Val Acc    : 64.31%
Best Val   : 64.31%
Label Smooth: 0.1
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.36 min


In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 2
# ============================================================

epoch = 2

print("\n" + "=" * 60)
print(f"REGULARIZATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(f"Label Smooth: {LABEL_SMOOTHING}")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


REGULARIZATION — EPOCH 2
Batch 50/661 | Loss: 1.5695
Batch 100/661 | Loss: 1.0465
Batch 150/661 | Loss: 0.8950
Batch 200/661 | Loss: 1.3034
Batch 250/661 | Loss: 0.9792
Batch 300/661 | Loss: 0.9366
Batch 350/661 | Loss: 2.6174
Batch 400/661 | Loss: 1.5080
Batch 450/661 | Loss: 0.7917
Batch 500/661 | Loss: 0.8011
Batch 550/661 | Loss: 1.7963
Batch 600/661 | Loss: 0.6600
Batch 650/661 | Loss: 0.6862

⭐ New best checkpoint saved

Train Loss : 1.1411
Train Acc  : 75.32%
Val Loss   : 1.0681
Val Acc    : 76.68%
Best Val   : 76.68%
Label Smooth: 0.1
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.44 min


In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 3
# ============================================================

epoch = 3

print("\n" + "=" * 60)
print(f"REGULARIZATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(f"Label Smooth: {LABEL_SMOOTHING}")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


REGULARIZATION — EPOCH 3
Batch 50/661 | Loss: 0.8565
Batch 100/661 | Loss: 1.6201
Batch 150/661 | Loss: 0.7339
Batch 200/661 | Loss: 1.2919
Batch 250/661 | Loss: 0.6624
Batch 300/661 | Loss: 0.9097
Batch 350/661 | Loss: 0.6158
Batch 400/661 | Loss: 0.5750
Batch 450/661 | Loss: 0.9878
Batch 500/661 | Loss: 0.6753
Batch 550/661 | Loss: 0.6120
Batch 600/661 | Loss: 0.9079
Batch 650/661 | Loss: 0.7734

⭐ New best checkpoint saved

Train Loss : 0.8476
Train Acc  : 90.61%
Val Loss   : 0.9413
Val Acc    : 82.33%
Best Val   : 82.33%
Label Smooth: 0.1
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.65 min


In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 4
# ============================================================

epoch = 4

print("\n" + "=" * 60)
print(f"REGULARIZATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(f"Label Smooth: {LABEL_SMOOTHING}")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


REGULARIZATION — EPOCH 4
Batch 50/661 | Loss: 0.6002
Batch 100/661 | Loss: 0.5898
Batch 150/661 | Loss: 0.7620
Batch 200/661 | Loss: 0.7088
Batch 250/661 | Loss: 0.5629
Batch 300/661 | Loss: 0.6056
Batch 350/661 | Loss: 0.7163
Batch 400/661 | Loss: 0.6035
Batch 450/661 | Loss: 0.5306
Batch 500/661 | Loss: 0.9400
Batch 550/661 | Loss: 0.6092
Batch 600/661 | Loss: 0.5650
Batch 650/661 | Loss: 0.8172

⭐ New best checkpoint saved

Train Loss : 0.7267
Train Acc  : 95.31%
Val Loss   : 0.8747
Val Acc    : 88.69%
Best Val   : 88.69%
Label Smooth: 0.1
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.44 min


In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 4 MILESTONE TEST
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------
# Load current BEST checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()


print("=" * 60)
print("REGULARIZATION — MILESTONE TEST")
print("=" * 60)

print("Best checkpoint epoch:", checkpoint["epoch"])

print(
    f"Best validation accuracy: "
    f"{checkpoint['best_val_acc'] * 100:.2f}%"
)


# ------------------------------------------------------------
# Standard CE only for reporting test loss
# ------------------------------------------------------------

test_criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# Test
# ------------------------------------------------------------

test_running_loss = 0.0
test_correct = 0
test_total = 0

all_labels = []
all_predictions = []


with torch.no_grad():

    for videos, labels in test_loader:

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)

            loss = test_criterion(
                outputs,
                labels
            )

        predictions = outputs.argmax(dim=1)

        test_running_loss += (
            loss.item() * labels.size(0)
        )

        test_correct += (
            predictions == labels
        ).sum().item()

        test_total += labels.size(0)

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

test_loss = test_running_loss / test_total
test_acc = test_correct / test_total


print("\n" + "=" * 60)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc * 100:.2f}%")

print(f"Correct   : {test_correct}")
print(f"Wrong     : {test_total - test_correct}")
print(f"Total     : {test_total}")

print("=" * 60)


print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=CLASSES,
        digits=4
    )
)


print("\nCONFUSION MATRIX\n")

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

REGULARIZATION — MILESTONE TEST
Best checkpoint epoch: 4
Best validation accuracy: 88.69%

Test Loss : 0.4707
Test Acc  : 87.32%
Correct   : 248
Wrong     : 36
Total     : 284

CLASSIFICATION REPORT

              precision    recall  f1-score   support

       cover     1.0000    0.7500    0.8571        28
     defense     0.9062    1.0000    0.9508        29
       flick     0.7931    0.8519    0.8214        27
        hook     0.9259    0.9259    0.9259        27
    late_cut     0.8077    0.7500    0.7778        28
      lofted     0.8750    0.9333    0.9032        30
        pull     0.8214    0.8519    0.8364        27
  square_cut     0.7857    0.7333    0.7586        30
    straight     0.9062    1.0000    0.9508        29
       sweep     0.9310    0.9310    0.9310        29

    accuracy                         0.8732       284
   macro avg     0.8752    0.8727    0.8713       284
weighted avg     0.8753    0.8732    0.8716       284


CONFUSION MATRIX

[[21  0  2  0  2  0  1

In [ ]:
import os

print("=" * 70)
print("SEARCHING FOR REGULARIZATION SAVED FILES")
print("=" * 70)

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if (
            "regularization" in file.lower()
            or file in [
                "regularization_latest.pth",
                "regularization_best.pth",
                "regularization_history.json",
                "regularization_config.json"
            ]
        ):
            print(os.path.join(root, file))

SEARCHING FOR REGULARIZATION SAVED FILES
/kaggle/input/notebooks/nivrithareddykunduru/video-swin-regularization/video_swin_regularization/regularization_history.json
/kaggle/input/notebooks/nivrithareddykunduru/video-swin-regularization/video_swin_regularization/regularization_best.pth
/kaggle/input/notebooks/nivrithareddykunduru/video-swin-regularization/video_swin_regularization/regularization_config.json
/kaggle/input/notebooks/nivrithareddykunduru/video-swin-regularization/video_swin_regularization/regularization_latest.pth


In [ ]:
import torch

device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda:0
GPU: Tesla T4


In [ ]:
import torch
import torch.nn as nn
from torchvision.models.video import swin3d_t, Swin3D_T_Weights

# ============================================================
# RECREATE MODEL
# ============================================================

weights = Swin3D_T_Weights.KINETICS400_V1

model = swin3d_t(weights=weights)

# Replace original Kinetics-400 classifier with our 10-class head
in_features = model.head.in_features

model.head = nn.Linear(
    in_features,
    10
)

model = model.to(device)

print("=" * 60)
print("MODEL RECREATED")
print("=" * 60)

print("Head input features:", in_features)
print("Number of classes  :", model.head.out_features)
print("Device             :", next(model.parameters()).device)

Downloading: "https://download.pytorch.org/models/swin3d_t-7615ae03.pth" to /root/.cache/torch/hub/checkpoints/swin3d_t-7615ae03.pth


100%|██████████| 122M/122M [00:00<00:00, 199MB/s] 


MODEL RECREATED
Head input features: 768
Number of classes  : 10
Device             : cuda:0


In [ ]:
# ============================================================
# RECREATE OPTIMIZER + LOSS + AMP SCALER
# ============================================================

BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1


# ------------------------------------------------------------
# Separate backbone and classification head parameters
# ------------------------------------------------------------

backbone_params = [
    param
    for name, param in model.named_parameters()
    if not name.startswith("head.")
]

head_params = list(
    model.head.parameters()
)


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR
        },
        {
            "params": head_params,
            "lr": HEAD_LR
        }
    ],
    weight_decay=WEIGHT_DECAY
)


# ------------------------------------------------------------
# Loss — same regularization experiment
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)


# ------------------------------------------------------------
# AMP scaler
# ------------------------------------------------------------

scaler = torch.amp.GradScaler("cuda")


print("=" * 60)
print("TRAINING COMPONENTS RECREATED")
print("=" * 60)

print("Optimizer       : AdamW")
print(f"Backbone LR     : {optimizer.param_groups[0]['lr']:.2e}")
print(f"Head LR         : {optimizer.param_groups[1]['lr']:.2e}")
print(f"Weight decay    : {WEIGHT_DECAY}")
print(f"Label smoothing : {LABEL_SMOOTHING}")
print(f"AMP enabled     : {scaler.is_enabled()}")

print(
    "Total optimized :",
    sum(
        p.numel()
        for group in optimizer.param_groups
        for p in group["params"]
    )
)

TRAINING COMPONENTS RECREATED
Optimizer       : AdamW
Backbone LR     : 1.00e-05
Head LR         : 1.00e-04
Weight decay    : 0.0001
Label smoothing : 0.1
AMP enabled     : True
Total optimized : 27858160


In [ ]:
import os
import json
import shutil
import torch

# ============================================================
# PATHS
# ============================================================

SAVED_DIR = (
    "/kaggle/input/notebooks/nivrithareddykunduru/"
    "video-swin-regularization/"
    "video_swin_regularization"
)

SOURCE_LATEST = os.path.join(
    SAVED_DIR,
    "regularization_latest.pth"
)

SOURCE_BEST = os.path.join(
    SAVED_DIR,
    "regularization_best.pth"
)

SAVE_DIR = "/kaggle/working/video_swin_regularization"
os.makedirs(SAVE_DIR, exist_ok=True)

LATEST_PATH = os.path.join(
    SAVE_DIR,
    "regularization_latest.pth"
)

BEST_PATH = os.path.join(
    SAVE_DIR,
    "regularization_best.pth"
)

HISTORY_PATH = os.path.join(
    SAVE_DIR,
    "regularization_history.json"
)

CONFIG_PATH = os.path.join(
    SAVE_DIR,
    "regularization_config.json"
)


# ============================================================
# LOAD CHECKPOINT
# ============================================================

checkpoint = torch.load(
    SOURCE_LATEST,
    map_location=device,
    weights_only=False
)


# ============================================================
# RESTORE MODEL
# ============================================================

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)


# ============================================================
# RESTORE OPTIMIZER
# ============================================================

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

# Move optimizer state tensors to GPU
for state in optimizer.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(device)


# ============================================================
# RESTORE AMP SCALER
# ============================================================

scaler.load_state_dict(
    checkpoint["scaler_state_dict"]
)


# ============================================================
# RESTORE HISTORY / CONFIG
# ============================================================

history = checkpoint["history"]
config = checkpoint["config"]

best_val_acc = checkpoint["best_val_acc"]

last_epoch = checkpoint["epoch"]
start_epoch = last_epoch + 1


# ============================================================
# COPY SAVED BEST INTO NEW WORKING DIRECTORY
# ============================================================

shutil.copy2(
    SOURCE_BEST,
    BEST_PATH
)


# ============================================================
# CREATE CURRENT LATEST CHECKPOINT COPY
# ============================================================

shutil.copy2(
    SOURCE_LATEST,
    LATEST_PATH
)


# ============================================================
# RESTORE JSON FILES
# ============================================================

with open(HISTORY_PATH, "w") as f:
    json.dump(history, f, indent=4)

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=4)


# ============================================================
# VERIFY
# ============================================================

print("\n" + "=" * 70)
print("REGULARIZATION CHECKPOINT RESTORED")
print("=" * 70)

print(f"Last completed epoch : {last_epoch}")
print(f"Next epoch           : {start_epoch}")

print(
    f"Best Val             : "
    f"{best_val_acc * 100:.2f}%"
)

print(
    f"Backbone LR          : "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"Head LR              : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(
    f"History epochs       : "
    f"{len(history['val_acc'])}"
)

print(
    "Validation history   :",
    [
        round(x * 100, 2)
        for x in history["val_acc"]
    ]
)

print(
    "Train history        :",
    [
        round(x * 100, 2)
        for x in history["train_acc"]
    ]
)

print(
    f"Best checkpoint ready: "
    f"{os.path.exists(BEST_PATH)}"
)

print(
    f"Latest ready         : "
    f"{os.path.exists(LATEST_PATH)}"
)

print("=" * 70)


REGULARIZATION CHECKPOINT RESTORED
Last completed epoch : 4
Next epoch           : 5
Best Val             : 88.69%
Backbone LR          : 1.00e-05
Head LR              : 1.00e-04
History epochs       : 4
Validation history   : [64.31, 76.68, 82.33, 88.69]
Train history        : [41.26, 75.32, 90.61, 95.31]
Best checkpoint ready: True
Latest ready         : True


In [ ]:
required = [
    "train_loader",
    "val_loader",
    "train_one_epoch",
    "validate",
    "save_training_state"
]

for name in required:
    print(f"{name:22s} :", name in globals())

train_loader           : False
val_loader             : False
train_one_epoch        : False
validate               : False
save_training_state    : False


In [ ]:
required = [
    "train_loader",
    "val_loader",
    "train_one_epoch",
    "validate",
    "save_training_state"
]

for name in required:
    print(f"{name:22s} :", name in globals())

train_loader           : False
val_loader             : False
train_one_epoch        : False
validate               : False
save_training_state    : False


In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import Swin3D_T_Weights


# ============================================================
# DATASET CONFIG
# ============================================================

DATASET_ROOT = (
    "/kaggle/input/datasets/nivrithareddykunduru/"
    "cricketshot-dataset/dataset"
)

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR   = os.path.join(DATASET_ROOT, "val")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")

CLASSES = [
    "cover",
    "defense",
    "flick",
    "hook",
    "late_cut",
    "lofted",
    "pull",
    "square_cut",
    "straight",
    "sweep"
]

CLASS_TO_IDX = {
    cls: idx
    for idx, cls in enumerate(CLASSES)
}

NUM_FRAMES = 32
BATCH_SIZE = 2


# ============================================================
# OFFICIAL SWIN PREPROCESSING
# ============================================================

weights = Swin3D_T_Weights.KINETICS400_V1

preprocess = weights.transforms()

train_transform = preprocess
val_transform   = preprocess
test_transform  = preprocess


# ============================================================
# UNIFORM TEMPORAL SAMPLING
# ============================================================

def uniform_temporal_indices(
    total_frames,
    num_frames=32
):

    if total_frames <= 0:
        return np.zeros(
            num_frames,
            dtype=np.int64
        )

    indices = np.linspace(
        0,
        total_frames - 1,
        num_frames
    )

    return np.round(
        indices
    ).astype(np.int64)


# ============================================================
# DATASET
# ============================================================

class CricketVideoDataset(Dataset):

    def __init__(
        self,
        root_dir,
        transform,
        num_frames=32
    ):

        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames

        self.samples = []

        valid_extensions = (
            ".avi",
            ".mp4",
            ".mov",
            ".mkv"
        )

        for class_name in CLASSES:

            class_dir = os.path.join(
                root_dir,
                class_name
            )

            if not os.path.isdir(class_dir):
                continue

            for file_name in sorted(
                os.listdir(class_dir)
            ):

                if file_name.lower().endswith(
                    valid_extensions
                ):

                    video_path = os.path.join(
                        class_dir,
                        file_name
                    )

                    label = CLASS_TO_IDX[
                        class_name
                    ]

                    self.samples.append(
                        (video_path, label)
                    )

        print(
            f"Loaded {len(self.samples)} videos "
            f"from {root_dir}"
        )


    def __len__(self):

        return len(self.samples)


    def __getitem__(self, idx):

        video_path, label = self.samples[idx]

        cap = cv2.VideoCapture(
            video_path
        )

        total_frames = int(
            cap.get(
                cv2.CAP_PROP_FRAME_COUNT
            )
        )

        indices = uniform_temporal_indices(
            total_frames,
            self.num_frames
        )

        frames = []

        for frame_idx in indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(frame_idx)
            )

            success, frame = cap.read()

            if success:

                frame = cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB
                )

                frames.append(frame)

        cap.release()


        if len(frames) == 0:

            raise RuntimeError(
                f"Could not decode video: "
                f"{video_path}"
            )


        # Pad if OpenCV failed to decode
        # some requested frames
        while len(frames) < self.num_frames:

            frames.append(
                frames[-1].copy()
            )


        frames = np.stack(
            frames[:self.num_frames]
        )


        frames = torch.from_numpy(
            frames
        ).permute(
            0, 3, 1, 2
        )


        frames = self.transform(
            frames
        )


        return frames, label


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = CricketVideoDataset(
    TRAIN_DIR,
    train_transform,
    NUM_FRAMES
)

val_dataset = CricketVideoDataset(
    VAL_DIR,
    val_transform,
    NUM_FRAMES
)

test_dataset = CricketVideoDataset(
    TEST_DIR,
    test_transform,
    NUM_FRAMES
)


# ============================================================
# CREATE LOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print("\n" + "=" * 60)
print("DATALOADERS RECREATED")
print("=" * 60)

print("Train videos :", len(train_dataset))
print("Val videos   :", len(val_dataset))
print("Test videos  :", len(test_dataset))

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

Loaded 1321 videos from /kaggle/input/datasets/nivrithareddykunduru/cricketshot-dataset/dataset/train
Loaded 283 videos from /kaggle/input/datasets/nivrithareddykunduru/cricketshot-dataset/dataset/val
Loaded 284 videos from /kaggle/input/datasets/nivrithareddykunduru/cricketshot-dataset/dataset/test

DATALOADERS RECREATED
Train videos : 1321
Val videos   : 283
Test videos  : 284
Train batches: 661
Val batches  : 142
Test batches : 142


In [ ]:
import time


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0


    for batch_idx, (videos, labels) in enumerate(
        loader,
        start=1
    ):

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)

            loss = criterion(
                outputs,
                labels
            )


        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()


        predictions = outputs.argmax(
            dim=1
        )


        batch_size = labels.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += batch_size


        if batch_idx % 50 == 0:

            print(
                f"Batch {batch_idx}/{len(loader)} "
                f"| Loss: {loss.item():.4f}"
            )


    epoch_loss = (
        running_loss / total
    )

    epoch_acc = (
        correct / total
    )


    return epoch_loss, epoch_acc



@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0


    for videos, labels in loader:

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)

            loss = criterion(
                outputs,
                labels
            )


        predictions = outputs.argmax(
            dim=1
        )

        batch_size = labels.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += batch_size


    epoch_loss = (
        running_loss / total
    )

    epoch_acc = (
        correct / total
    )


    return epoch_loss, epoch_acc


print("✅ train_one_epoch() recreated")
print("✅ validate() recreated")

✅ train_one_epoch() recreated
✅ validate() recreated


In [ ]:
import json


def save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=False
):

    checkpoint = {

        "epoch":
            epoch,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "best_val_acc":
            best_val_acc,

        "history":
            history,

        "config":
            config,

        "classes":
            CLASSES
    }


    # Always save latest
    torch.save(
        checkpoint,
        LATEST_PATH
    )


    # Save best only when validation improves
    if is_best:

        torch.save(
            checkpoint,
            BEST_PATH
        )


    with open(
        HISTORY_PATH,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=4
        )


    with open(
        CONFIG_PATH,
        "w"
    ) as f:

        json.dump(
            config,
            f,
            indent=4
        )


print("✅ save_training_state() recreated")

✅ save_training_state() recreated


In [ ]:
required = [
    "train_loader",
    "val_loader",
    "train_one_epoch",
    "validate",
    "save_training_state"
]

print("=" * 50)

for name in required:
    print(
        f"{name:22s} :",
        name in globals()
    )

print("=" * 50)

# Make sure restored state was NOT overwritten
print("Restored best val :", round(best_val_acc * 100, 2))
print("History epochs    :", len(history["val_acc"]))

train_loader           : True
val_loader             : True
train_one_epoch        : True
validate               : True
save_training_state    : True
Restored best val : 88.69
History epochs    : 4


In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 5
# ============================================================

epoch = 5

print("\n" + "=" * 60)
print(f"REGULARIZATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


# ------------------------------------------------------------
# VALIDATE
# ------------------------------------------------------------

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


# ------------------------------------------------------------
# UPDATE HISTORY
# ------------------------------------------------------------

history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


# ------------------------------------------------------------
# CHECK BEST
# ------------------------------------------------------------

is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

elapsed_minutes = (time.time() - start_time) / 60

print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(f"Label Smooth: {LABEL_SMOOTHING}")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


REGULARIZATION — EPOCH 5
Batch 50/661 | Loss: 0.6394
Batch 100/661 | Loss: 0.6458
Batch 150/661 | Loss: 0.5778
Batch 200/661 | Loss: 0.5674
Batch 250/661 | Loss: 0.5480
Batch 300/661 | Loss: 0.5611
Batch 350/661 | Loss: 0.5614
Batch 400/661 | Loss: 0.5956
Batch 450/661 | Loss: 0.6403
Batch 500/661 | Loss: 0.7422
Batch 550/661 | Loss: 0.5756
Batch 600/661 | Loss: 0.5477
Batch 650/661 | Loss: 0.5544

⭐ New best checkpoint saved

Train Loss : 0.6570
Train Acc  : 97.73%
Val Loss   : 0.8506
Val Acc    : 90.11%
Best Val   : 90.11%
Label Smooth: 0.1
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.09 min


In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 6
# ============================================================

epoch = 6

print("\n" + "=" * 60)
print(f"REGULARIZATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()

# TRAIN
train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)

# VALIDATE
val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)

# UPDATE HISTORY
history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)

# CHECK BEST
is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")

# SAVE
save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)

elapsed_minutes = (time.time() - start_time) / 60

# RESULTS
print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(f"Label Smooth: {LABEL_SMOOTHING}")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


REGULARIZATION — EPOCH 6
Batch 50/661 | Loss: 0.6420
Batch 100/661 | Loss: 0.8593
Batch 150/661 | Loss: 0.5397
Batch 200/661 | Loss: 0.5451
Batch 250/661 | Loss: 0.5317
Batch 300/661 | Loss: 0.5298
Batch 350/661 | Loss: 0.5385
Batch 400/661 | Loss: 0.6810
Batch 450/661 | Loss: 0.5407
Batch 500/661 | Loss: 0.6923
Batch 550/661 | Loss: 0.6955
Batch 600/661 | Loss: 0.5560
Batch 650/661 | Loss: 0.5518

Train Loss : 0.6186
Train Acc  : 98.86%
Val Loss   : 0.8520
Val Acc    : 87.28%
Best Val   : 90.11%
Label Smooth: 0.1
LR backbone: 1.00e-05
LR head    : 1.00e-04
Time       : 21.33 min


In [ ]:
# ============================================================
# REGULARIZATION — BEST CHECKPOINT TEST
# Expected best: Epoch 5, Val Acc = 90.11%
# ============================================================

import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix


# ------------------------------------------------------------
# 1. Load BEST checkpoint
# ------------------------------------------------------------

best_checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False
)

print("=" * 65)
print("REGULARIZATION — BEST CHECKPOINT TEST")
print("=" * 65)

print(
    "Best checkpoint epoch :",
    best_checkpoint["epoch"]
)

print(
    f"Best validation acc   : "
    f"{best_checkpoint['best_val_acc'] * 100:.2f}%"
)


# Safety check
assert best_checkpoint["epoch"] == 5, (
    f"Expected Epoch 5 best checkpoint, "
    f"but found Epoch {best_checkpoint['epoch']}"
)


# ------------------------------------------------------------
# 2. Load best model weights
# ------------------------------------------------------------

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()


# ------------------------------------------------------------
# 3. Standard Cross Entropy for reporting test loss
# ------------------------------------------------------------

test_criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 4. TEST
# ------------------------------------------------------------

test_running_loss = 0.0
test_correct = 0
test_total = 0

all_labels = []
all_predictions = []


with torch.no_grad():

    for videos, labels in test_loader:

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)

            loss = test_criterion(
                outputs,
                labels
            )


        predictions = outputs.argmax(dim=1)

        batch_size = labels.size(0)

        test_running_loss += (
            loss.item() * batch_size
        )

        test_correct += (
            predictions == labels
        ).sum().item()

        test_total += batch_size


        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )


# ------------------------------------------------------------
# 5. RESULTS
# ------------------------------------------------------------

test_loss = test_running_loss / test_total

test_acc = test_correct / test_total


print("\n" + "=" * 65)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc * 100:.2f}%")

print(f"Correct   : {test_correct}")
print(f"Wrong     : {test_total - test_correct}")
print(f"Total     : {test_total}")

print("=" * 65)


# ------------------------------------------------------------
# 6. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=CLASSES,
        digits=4
    )
)


# ------------------------------------------------------------
# 7. CONFUSION MATRIX
# ------------------------------------------------------------

print("\nCONFUSION MATRIX\n")

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

REGULARIZATION — BEST CHECKPOINT TEST
Best checkpoint epoch : 5
Best validation acc   : 90.11%

Test Loss : 0.4258
Test Acc  : 88.38%
Correct   : 251
Wrong     : 33
Total     : 284

CLASSIFICATION REPORT

              precision    recall  f1-score   support

       cover     0.8696    0.7143    0.7843        28
     defense     0.9355    1.0000    0.9667        29
       flick     0.8276    0.8889    0.8571        27
        hook     0.8966    0.9630    0.9286        27
    late_cut     0.8214    0.8214    0.8214        28
      lofted     0.8286    0.9667    0.8923        30
        pull     0.9200    0.8519    0.8846        27
  square_cut     0.7857    0.7333    0.7586        30
    straight     1.0000    1.0000    1.0000        29
       sweep     0.9630    0.8966    0.9286        29

    accuracy                         0.8838       284
   macro avg     0.8848    0.8836    0.8822       284
weighted avg     0.8846    0.8838    0.8822       284


CONFUSION MATRIX

[[20  0  3  0  2 

In [ ]:
# ============================================================
# RESTORE LATEST — EPOCH 6
# ============================================================

latest_checkpoint = torch.load(
    LATEST_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint epoch:", latest_checkpoint["epoch"])

model.load_state_dict(
    latest_checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    latest_checkpoint["optimizer_state_dict"]
)

for state in optimizer.state.values():
    for key, value in state.items():
        if torch.is_tensor(value):
            state[key] = value.to(device)

scaler.load_state_dict(
    latest_checkpoint["scaler_state_dict"]
)

history = latest_checkpoint["history"]
best_val_acc = latest_checkpoint["best_val_acc"]

print(f"Best Val      : {best_val_acc * 100:.2f}%")
print(f"History epochs: {len(history['val_acc'])}")
print(f"Backbone LR   : {optimizer.param_groups[0]['lr']:.2e}")
print(f"Head LR       : {optimizer.param_groups[1]['lr']:.2e}")

print("✅ Epoch 6 training state restored")

Checkpoint epoch: 6
Best Val      : 90.11%
History epochs: 6
Backbone LR   : 1.00e-05
Head LR       : 1.00e-04
✅ Epoch 6 training state restored


In [ ]:
# ============================================================
# LR REDUCTION BEFORE EPOCH 7
# ============================================================

optimizer.param_groups[0]["lr"] *= 0.5
optimizer.param_groups[1]["lr"] *= 0.5

print(
    "New backbone LR:",
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    "New head LR    :",
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

New backbone LR: 5.00e-06
New head LR    : 5.00e-05


In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 7
# ============================================================

epoch = 7

print("\n" + "=" * 60)
print(f"REGULARIZATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()

train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)

val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)

history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)

is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")

save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)

elapsed_minutes = (time.time() - start_time) / 60

print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")
print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(f"Label Smooth: {LABEL_SMOOTHING}")
print(f"LR backbone: {optimizer.param_groups[0]['lr']:.2e}")
print(f"LR head    : {optimizer.param_groups[1]['lr']:.2e}")
print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


REGULARIZATION — EPOCH 7
Batch 50/661 | Loss: 0.6832
Batch 100/661 | Loss: 0.5479
Batch 150/661 | Loss: 0.5299
Batch 200/661 | Loss: 0.5397
Batch 250/661 | Loss: 0.6113
Batch 300/661 | Loss: 1.2888
Batch 350/661 | Loss: 0.5250
Batch 400/661 | Loss: 0.5250
Batch 450/661 | Loss: 0.5313
Batch 500/661 | Loss: 0.5471
Batch 550/661 | Loss: 0.5696
Batch 600/661 | Loss: 0.5412
Batch 650/661 | Loss: 0.5463

Train Loss : 0.5909
Train Acc  : 99.17%
Val Loss   : 0.8343
Val Acc    : 89.40%
Best Val   : 90.11%
Label Smooth: 0.1
LR backbone: 5.00e-06
LR head    : 5.00e-05
Time       : 21.88 min


In [ ]:
# ============================================================
# REGULARIZATION — EPOCH 8
# FINAL PLANNED EPOCH
# ============================================================

epoch = 8

print("\n" + "=" * 60)
print(f"REGULARIZATION — EPOCH {epoch}")
print("=" * 60)

start_time = time.time()


# TRAIN
train_loss, train_acc = train_one_epoch(
    model,
    train_loader,
    optimizer,
    criterion,
    scaler,
    device
)


# VALIDATE
val_loss, val_acc = validate(
    model,
    val_loader,
    criterion,
    device
)


# UPDATE HISTORY
history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)


# CHECK BEST
is_best = val_acc > best_val_acc

if is_best:
    best_val_acc = val_acc
    print("\n⭐ New best checkpoint saved")


# SAVE
save_training_state(
    epoch,
    model,
    optimizer,
    scaler,
    best_val_acc,
    history,
    config,
    is_best=is_best
)


elapsed_minutes = (time.time() - start_time) / 60


# RESULTS
print("\n" + "=" * 60)

print(f"Train Loss : {train_loss:.4f}")
print(f"Train Acc  : {train_acc * 100:.2f}%")

print(f"Val Loss   : {val_loss:.4f}")
print(f"Val Acc    : {val_acc * 100:.2f}%")

print(f"Best Val   : {best_val_acc * 100:.2f}%")

print(f"Label Smooth: {LABEL_SMOOTHING}")

print(
    f"LR backbone: "
    f"{optimizer.param_groups[0]['lr']:.2e}"
)

print(
    f"LR head    : "
    f"{optimizer.param_groups[1]['lr']:.2e}"
)

print(f"Time       : {elapsed_minutes:.2f} min")

print("=" * 60)


REGULARIZATION — EPOCH 8
Batch 50/661 | Loss: 0.5712
Batch 100/661 | Loss: 0.5945
Batch 150/661 | Loss: 0.5393
Batch 200/661 | Loss: 0.5501
Batch 250/661 | Loss: 0.7052
Batch 300/661 | Loss: 0.6263
Batch 350/661 | Loss: 0.5958
Batch 400/661 | Loss: 0.5785
Batch 450/661 | Loss: 0.5448
Batch 500/661 | Loss: 1.0702
Batch 550/661 | Loss: 0.6436
Batch 600/661 | Loss: 0.5862
Batch 650/661 | Loss: 0.5526

Train Loss : 0.5860
Train Acc  : 98.86%
Val Loss   : 0.8321
Val Acc    : 87.63%
Best Val   : 90.11%
Label Smooth: 0.1
LR backbone: 5.00e-06
LR head    : 5.00e-05
Time       : 21.36 min


In [ ]:
# ============================================================
# REGULARIZATION EXPERIMENT — FINAL TEST
# ============================================================

import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix


# ------------------------------------------------------------
# 1. LOAD BEST CHECKPOINT
# ------------------------------------------------------------

best_checkpoint = torch.load(
    BEST_PATH,
    map_location=device,
    weights_only=False
)

best_epoch = best_checkpoint["epoch"]
best_val = best_checkpoint["best_val_acc"]


print("=" * 70)
print("REGULARIZATION — FINAL TEST")
print("=" * 70)

print(f"Selected epoch : {best_epoch}")
print(f"Best Val Acc   : {best_val * 100:.2f}%")



# ------------------------------------------------------------
# 2. LOAD BEST MODEL WEIGHTS
# ------------------------------------------------------------

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()


# ------------------------------------------------------------
# 3. STANDARD CROSS ENTROPY FOR TEST REPORTING
# ------------------------------------------------------------

test_criterion = nn.CrossEntropyLoss()


# ------------------------------------------------------------
# 4. TEST
# ------------------------------------------------------------

test_running_loss = 0.0
test_correct = 0
test_total = 0

all_labels = []
all_predictions = []


with torch.no_grad():

    for videos, labels in test_loader:

        videos = videos.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )


        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            outputs = model(videos)

            loss = test_criterion(
                outputs,
                labels
            )


        predictions = outputs.argmax(
            dim=1
        )

        batch_size = labels.size(0)

        test_running_loss += (
            loss.item() * batch_size
        )

        test_correct += (
            predictions == labels
        ).sum().item()

        test_total += batch_size


        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )


# ------------------------------------------------------------
# 5. FINAL METRICS
# ------------------------------------------------------------

test_loss = (
    test_running_loss / test_total
)

test_acc = (
    test_correct / test_total
)


print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc * 100:.2f}%")

print(f"Correct   : {test_correct}")
print(f"Wrong     : {test_total - test_correct}")
print(f"Total     : {test_total}")

print("=" * 70)



# ------------------------------------------------------------
# 6. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nCLASSIFICATION REPORT\n")

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=CLASSES,
        digits=4
    )
)



# ------------------------------------------------------------
# 7. CONFUSION MATRIX
# ------------------------------------------------------------

print("\nCONFUSION MATRIX\n")

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print(cm)

REGULARIZATION — FINAL TEST
Selected epoch : 5
Best Val Acc   : 90.11%

FINAL RESULTS
Test Loss : 0.4258
Test Acc  : 88.38%
Correct   : 251
Wrong     : 33
Total     : 284

CLASSIFICATION REPORT

              precision    recall  f1-score   support

       cover     0.8696    0.7143    0.7843        28
     defense     0.9355    1.0000    0.9667        29
       flick     0.8276    0.8889    0.8571        27
        hook     0.8966    0.9630    0.9286        27
    late_cut     0.8214    0.8214    0.8214        28
      lofted     0.8286    0.9667    0.8923        30
        pull     0.9200    0.8519    0.8846        27
  square_cut     0.7857    0.7333    0.7586        30
    straight     1.0000    1.0000    1.0000        29
       sweep     0.9630    0.8966    0.9286        29

    accuracy                         0.8838       284
   macro avg     0.8848    0.8836    0.8822       284
weighted avg     0.8846    0.8838    0.8822       284


CONFUSION MATRIX

[[20  0  3  0  2  0  1  2  